# GreenLedger — Campus Resource Optimisation (engine, real data only)
### Innovators Conclave 2026 · IC26-05-01

Everything in this notebook runs on **real Building Data Genome 2 (BDG2) meter data**. Nothing is generated, injected or interpolated, there is no keyword matching, and every parameter is listed with its justification in cell 1.

**What it does**
1. **Picks the campus:** the BDG2 site that meters the most resource types (electricity, chilled water, steam, hot water, gas, water, irrigation, solar). Every building on that site is included if its meter passes a data-quality rule.
2. **Models normal use:** each meter is predicted from its peer meters, the calendar and the weather.
3. **Raises alerts:** a meter is flagged when it does something more extreme than anything it did in 18 months of out-of-sample history. Meter faults (no data or zero readings) are listed separately.
4. **Verifies actions:** a counterfactual from comparison meters, placebo tests in time and space, robust 95% intervals and a permutation p-value. Savings count only at their lower bound.
5. **Audits itself:** model accuracy on six unseen months, and how often the verifier certifies a change at dates where no action was logged.
6. **Exports:** a report, plus a labelled dataset of real readings with the platform's own labels.

**Run in Google Colab**
1. Upload your BDG2 zip (for example `bdg2.zip`) using the Files panel on the left. It should contain `metadata.csv`, `weather.csv` and the `*_cleaned.csv` meter files.
2. Click *Runtime → Run all*. Runtime is about 10–15 minutes.
3. The last cell prints a public link to a working app (an interim UI; the final UI comes next).
4. **Send back the output of the "Results summary" cell** (plus any error message).

In [ ]:
!pip -q install lightgbm plotly gradio scipy

## 1 · Setup, data location and parameters
Finds and unzips the BDG2 files automatically. Prints every parameter and why it has that value.

In [ ]:
# ============================================================
# 1 · SETUP, DATA LOCATION AND PARAMETERS
# Every parameter below is either a statistical convention, a natural time period,
# a published external factor, a library default, or derived from the data.
# The table printed at the end of this cell lists each one and its justification.
# ============================================================
import os, io, json, glob, time, math, sqlite3, zipfile, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import lightgbm as lgb
import plotly.graph_objects as go
from scipy import stats

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Paths: works on Kaggle, Google Colab and locally ----------
# Colab: just upload the BDG2 .zip (Files panel, or Google Drive) — it is found and unzipped automatically.
# Optional: set BDG2_ZIP to the zip's exact path if it lives somewhere unusual.
BDG2_ZIP = os.environ.get("BDG2_ZIP", "")
IN_COLAB = os.path.isdir("/content")
OUT_DIR = os.environ.get("GL_OUT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working")
                         else ("/content/greenledger_out" if IN_COLAB else "./greenledger_out"))
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, "greenledger.db")
UNZIP_DIR = os.path.join(OUT_DIR, "bdg2_data")
SEARCH_ROOTS = [os.environ.get("GL_DATA_DIR", ""), UNZIP_DIR, "/kaggle/input", "/content", "./data", "."]
METER_TYPES = ["electricity", "chilledwater", "steam", "hotwater", "gas", "water", "irrigation", "solar"]
NEEDED = ["metadata.csv", "weather.csv"] + [f"{m}_cleaned.csv" for m in METER_TYPES]


def find_file(name):
    """Find a BDG2 file anywhere under the search roots; skip Git-LFS pointer stubs (<1 KB)."""
    for root in SEARCH_ROOTS:
        if not root or not os.path.isdir(root):
            continue
        hits = [os.path.join(root, name)] + glob.glob(os.path.join(root, "**", name), recursive=True)
        hits = [h for h in hits if os.path.isfile(h) and os.path.getsize(h) > 1024]
        if hits:
            return sorted(hits, key=len)[0]
    return None


def _extract_bdg2_zip():
    """Find a zip containing the BDG2 CSVs and extract only the 4 files we need (nested zips handled)."""
    cands = [BDG2_ZIP] if BDG2_ZIP else []
    for root in (".", "/content", "/content/drive/MyDrive", "/kaggle/input"):
        cands += glob.glob(os.path.join(root, "*.zip")) + glob.glob(os.path.join(root, "*", "*.zip"))
    seen, queue, n_inner = set(), [c for c in cands if c and os.path.isfile(c)], 0
    while queue:
        z = queue.pop(0)
        if z in seen:
            continue
        seen.add(z)
        try:
            with zipfile.ZipFile(z) as zf:
                names = zf.namelist()
                got = 0
                for want in NEEDED:
                    m = sorted([n for n in names if n.split("/")[-1] == want and "/raw/" not in n], key=len)
                    if not m or zf.getinfo(m[0]).file_size <= 1024:      # skip Git-LFS pointer stubs
                        continue
                    os.makedirs(UNZIP_DIR, exist_ok=True)
                    with zf.open(m[0]) as src, open(os.path.join(UNZIP_DIR, want), "wb") as dst:
                        while True:
                            chunk = src.read(1 << 22)
                            if not chunk:
                                break
                            dst.write(chunk)
                    got += 1
                for inner in [n for n in names if n.lower().endswith(".zip")]:   # zip inside a zip
                    n_inner += 1
                    tgt = os.path.join(OUT_DIR, f"_inner_{n_inner}.zip")
                    with zf.open(inner) as src, open(tgt, "wb") as dst:
                        while True:
                            chunk = src.read(1 << 22)
                            if not chunk:
                                break
                            dst.write(chunk)
                    queue.append(tgt)
                if got:
                    print(f"Extracted {got} BDG2 file(s) from {z} → {UNZIP_DIR}")
        except zipfile.BadZipFile:
            print("Skipping unreadable zip:", z)


if not all(find_file(n) for n in NEEDED):
    _extract_bdg2_zip()
for _n in NEEDED:
    _p = find_file(_n)
    print(f"  {_n:26s} → {_p or 'NOT FOUND'}" + (f"  ({os.path.getsize(_p) / 1e6:,.0f} MB)" if _p else ""))
_stub = [h for r in SEARCH_ROOTS if r and os.path.isdir(r)
         for h in glob.glob(os.path.join(r, "**", "electricity_cleaned.csv"), recursive=True) if os.path.getsize(h) <= 1024]
if _stub and not find_file("electricity_cleaned.csv"):
    raise RuntimeError("Your BDG2 files are Git-LFS pointer stubs (a few hundred bytes), not the real CSVs. This "
                       "happens with GitHub's 'Download ZIP'. Download the data from Kaggle "
                       "('buildingdatagenomeproject2') or the BDG2 GitHub 'data' links instead.")



# ---------- Parameters (no tuned or invented values) ----------
CI_LEVEL = 0.95                                   # statistical convention
ALPHA = 1 - CI_LEVEL                              # 5% significance
MIN_PLACEBOS = math.ceil(1 / ALPHA) - 1           # = 19: fewest placebos for which a permutation p-value can reach α
TRAIN_DAYS = 365                                  # one full annual cycle (seasons + academic calendar)
POST_WINDOWS = [28, 42, 56]                       # 4 / 6 / 8 weeks — user-selectable measurement windows
PLACEBO_STEP_DAYS = 1                             # every eligible date (windows are whole weeks → identical weekday mix)
COMPLETENESS = 0.90                               # data-quality rule: ≥90% valid readings
MONITOR_MONTHS = 6                                # last 6 calendar months = monitoring period; all earlier data = training
SITE_OVERRIDE = None                              # None = automatic (campus with the most metered resource types);
                                                  # or e.g. "Panther" (irrigation) / "Eagle" (steam)
EMISSION_FACTOR = 0.710                           # kgCO2/kWh grid electricity — CEA CO2 Baseline Database v21.0, FY2024-25
EF_SOURCE = "CEA CO₂ Baseline Database v21.0 (FY2024-25, provisional) · 0.710 kgCO₂/kWh"
GAS_EF = 56100 / 277777.8                         # kgCO2/kWh natural gas = 56,100 kgCO2/TJ (IPCC 2006 Guidelines, Vol.2 Tab.2.2)
GAS_EF_SOURCE = "IPCC 2006 Guidelines default for natural gas · 56,100 kgCO₂/TJ = 0.202 kgCO₂/kWh"
TARIFF_INR_PER_KWH = None                         # set your campus's actual tariff to show ₹ values; None = no ₹ shown
LGB_PARAMS = dict(random_state=0, verbosity=-1)   # LightGBM library defaults (no tuning)
LGB_SMALL = dict(LGB_PARAMS, n_jobs=1)            # same model; single thread for small daily fits (speed only)

PARAMETER_TABLE = pd.DataFrame([
    ("Confidence level", "95%", "Statistical convention"),
    ("Significance α", "5%", "= 1 − confidence level"),
    ("Minimum placebo windows", str(MIN_PLACEBOS), "Smallest K for which a permutation p-value (1/(K+1)) can be ≤ α"),
    ("Baseline training window", "365 days", "One full annual cycle"),
    ("Measurement windows", "28 / 42 / 56 days", "4, 6 or 8 whole weeks — chosen by the user in the app"),
    ("Placebo dates", "every eligible day", "Windows are whole weeks, so every start date has the same weekday mix"),
    ("Spread estimator", "median absolute deviation × 1.4826", "Standard robust estimator (equals the SD for normal data; "
     "resists placebos contaminated by real events)"),
    ("Data completeness rule", "90%", "Meters, days, training and measurement windows need ≥90% of readings present"),
    ("Monitoring period", "last 6 calendar months", "Everything earlier (≥12 months) trains the models"),
    ("Alert thresholds", "data-derived", "Largest out-of-sample excess seen in the training period, per meter and per scale"),
    ("Alert scales", "1 hour · 1 day · 1 week · daily minimum", "Natural time units; daily minimum = base load / night flow"),
    ("Grid emission factor", "0.710 kgCO₂/kWh", "CEA CO₂ Baseline Database v21.0 (published)"),
    ("Natural-gas emission factor", "0.202 kgCO₂/kWh", "IPCC 2006 Guidelines default, 56,100 kgCO₂/TJ (published)"),
    ("Chilled water / hot water / steam CO₂", "not computed", "Depends on the campus central plant — no factor is invented"),
    ("Campus (site)", "automatic", "BDG2 site with the most metered resource types; override with SITE_OVERRIDE"),
    ("Tariff", "not set", "No ₹ figures unless the campus enters its real tariff"),
    ("Model hyper-parameters", "LightGBM defaults", "No tuning"),
], columns=["Parameter", "Value", "Why this value"])
print(PARAMETER_TABLE.to_string(index=False))

## 2 · Real meter data: campus selection and data quality
Campus = the BDG2 site with the most metered resource types. Meter faults (missing or zero days) are detected and excluded.

In [ ]:
# ============================================================
# 2 · REAL METER DATA (Building Data Genome 2) — every metered campus resource
# BDG2 meters: electricity, chilled water, steam, hot water, natural gas, water,
# irrigation and solar generation. Nothing is generated or filled in.
#   • a meter qualifies if ≥90% of its hourly readings are present and it is not constant
#   • the campus = the BDG2 site with the MOST metered resource types (ties → more meters),
#     all building types included (classrooms, offices, residences, labs, assembly halls …)
# ============================================================
P_META, P_WX = find_file("metadata.csv"), find_file("weather.csv")
assert P_META, ("BDG2 files not found. Kaggle: attach the Building Data Genome 2 dataset. "
                "Colab: upload the BDG2 .zip to /content (Files panel) or set BDG2_ZIP to its path, then re-run cell 1.")
meta = pd.read_csv(P_META).set_index("building_id")

RESOURCES = {
    "electricity":  {"label": "Electricity",          "unit": "kWh", "kind": "consumption", "color": "#10B981"},
    "chilledwater": {"label": "Chilled water (cooling)", "unit": "kWh", "kind": "consumption", "color": "#22D3EE"},
    "steam":        {"label": "Steam",                 "unit": "kWh", "kind": "consumption", "color": "#A78BFA"},
    "hotwater":     {"label": "Hot water (heating)",   "unit": "kWh", "kind": "consumption", "color": "#F472B6"},
    "gas":          {"label": "Natural gas",           "unit": "kWh", "kind": "consumption", "color": "#FB923C"},
    "water":        {"label": "Water",                 "unit": "L",   "kind": "consumption", "color": "#3B82F6"},
    "irrigation":   {"label": "Irrigation water",      "unit": "L",   "kind": "consumption", "color": "#84CC16"},
    "solar":        {"label": "Solar generation",      "unit": "kWh", "kind": "generation",  "color": "#FACC15"},
    "waste":        {"label": "Waste",                 "unit": "kg",  "kind": "consumption", "color": "#F59E0B"},
}
SIGN = {r: (-1 if v["kind"] == "generation" else 1) for r, v in RESOURCES.items()}   # +1: more is worse

# ---------- meter quality screen across ALL sites, on the TRAINING period only ----------
# A meter qualifies if, in training: ≥90% of hours present, ≥90% of days with non-zero use
# (BDG2 records an offline meter as zeros), and readings are not constant.
_raw = {}
for mt in METER_TYPES:
    p = find_file(f"{mt}_cleaned.csv")
    if p:
        d = pd.read_csv(p, parse_dates=["timestamp"], index_col="timestamp").apply(pd.to_numeric, errors="coerce")
        _raw[mt] = d.where(d >= 0)                              # negative readings are invalid → missing
IDX = pd.date_range(min(d.index.min() for d in _raw.values()), max(d.index.max() for d in _raw.values()), freq="h")
DATA_END = IDX.max().normalize()
_ev = DATA_END - pd.DateOffset(months=MONITOR_MONTHS) + pd.Timedelta(days=1)
EVAL_START = pd.Timestamp(_ev.year, _ev.month, 1)
TRAIN_END = EVAL_START - pd.Timedelta(hours=1)
DAYS = pd.date_range(IDX.min().normalize(), DATA_END, freq="D")

_q = []
for mt, d in _raw.items():
    tr = d.reindex(IDX).loc[:TRAIN_END]
    pres = tr.notna().mean()
    dsum = tr.resample("D").sum(min_count=1)
    active = (dsum > 0).mean()
    const = tr.std() == 0
    for b in d.columns:
        if b in meta.index:
            _q.append((meta.loc[b, "site_id"], mt, b,
                       bool(pres[b] >= COMPLETENESS and active[b] >= COMPLETENESS and not const[b])))
QUAL = pd.DataFrame(_q, columns=["site", "resource", "building", "ok"])
_per_site = QUAL[QUAL.ok].groupby(["site", "resource"]).size().unstack(fill_value=0)
_rank = pd.DataFrame({"types": (_per_site > 0).sum(axis=1), "meters": _per_site.sum(axis=1)})
SITE = SITE_OVERRIDE or _rank.sort_values(["types", "meters"], ascending=False).index[0]
print(f"Campus = BDG2 site '{SITE}': {int(_rank.loc[SITE, 'types'])} metered resource types, "
      f"{int(_rank.loc[SITE, 'meters'])} qualifying meters")


def short(bid):
    return bid.split("_")[-1]                 # real BDG2 building name, e.g. 'Bobcat_education_Angela' → 'Angela'


METERS, BLDS, FREQ, BLD_USE = {}, {}, {}, {}
for mt, d in _raw.items():
    ids = sorted(QUAL[(QUAL.site == SITE) & (QUAL.resource == mt) & QUAL.ok].building)
    if not ids:
        continue
    df = d.reindex(IDX)[ids]
    df.columns = [short(c) for c in ids]
    METERS[mt], BLDS[mt], FREQ[mt] = df, list(df.columns), "h"
    for c in ids:
        BLD_USE[short(c)] = meta.loc[c, "primaryspaceusage"]
del _raw

if P_WX:
    _wx = pd.read_csv(P_WX, parse_dates=["timestamp"])
    _wx = _wx[_wx.site_id == SITE].drop_duplicates("timestamp").set_index("timestamp")
    TEMP_H = _wx["airTemperature"].reindex(IDX)           # gaps stay missing; LightGBM handles them
else:
    TEMP_H = pd.Series(np.nan, index=IDX)
TEMP_D = TEMP_H.resample("D").mean()

# ---------- optional REAL waste log (no BDG2 site measures waste) ----------
# waste.csv with columns: date, building, kg   (building = BDG2 name or full id)
P_WASTE = find_file("waste.csv")
if P_WASTE:
    _ws = pd.read_csv(P_WASTE, parse_dates=["date"])
    _ws["building"] = _ws["building"].astype(str).map(short)
    _wd = _ws.pivot_table(index="date", columns="building", values="kg", aggfunc="sum").reindex(DAYS)
    _wt = _wd.loc[:TRAIN_END]
    _wd = _wd[[c for c in _wd.columns if _wt[c].notna().mean() >= COMPLETENESS
               and (_wt[c] > 0).mean() >= COMPLETENESS and _wt[c].std() > 0]]
    if _wd.shape[1]:
        METERS["waste"], BLDS["waste"], FREQ["waste"] = _wd, list(_wd.columns), "D"

RES_LIST = [r for r in RESOURCES if r in METERS]

# ---------- daily totals (a day counts if ≥90% of its hours are present: mean × 24) ----------
DAILY = {}
for r in RES_LIST:
    for b in BLDS[r]:
        s = METERS[r][b]
        if FREQ[r] == "h":
            n = s.resample("D").count()
            DAILY[(r, b)] = (s.resample("D").mean() * 24).where(n >= COMPLETENESS * 24).reindex(DAYS)
        else:
            DAILY[(r, b)] = s.reindex(DAYS)


# ---------- meter faults: days with <90% of hours present, or a zero total, while the meter was
# active in training. They are reported as data faults and excluded from every other test.
FAULT_DAYS = {}
for r in RES_LIST:
    for b in BLDS[r]:
        s_ = METERS[r][b]
        if FREQ[r] == "h":
            n = s_.resample("D").count().reindex(DAYS)
            tot = s_.resample("D").sum(min_count=1).reindex(DAYS)
            bad = (n < COMPLETENESS * 24) | (tot == 0)
        else:
            bad = s_.reindex(DAYS).isna() | (s_.reindex(DAYS) == 0)
        FAULT_DAYS[(r, b)] = set(DAYS[bad.values & (DAYS >= EVAL_START)])
        bad_days = DAYS[bad.values]
        if FREQ[r] == "h":                                          # fault days carry no information
            METERS[r].loc[METERS[r].index.normalize().isin(bad_days), b] = np.nan
        else:
            METERS[r].loc[METERS[r].index.isin(bad_days), b] = np.nan
        DAILY[(r, b)] = DAILY[(r, b)].where(~bad.values)


def daily(res, b):
    return DAILY[(res, b)].copy()


# ---------- resource coverage: every campus resource, metered or not ----------
_other = QUAL[QUAL.ok].groupby("resource").site.apply(lambda s: ", ".join(sorted(set(s) - {SITE}))).to_dict()
COVERAGE = pd.DataFrame([{
    "Resource": RESOURCES[r]["label"],
    "Status on this campus": (f"metered — {len(BLDS[r])} meter{'s' if len(BLDS[r]) != 1 else ''}" if r in METERS else
                              ("not measured in BDG2 — upload a real waste.csv" if r == "waste" else "not metered on this campus")),
    "Other BDG2 campuses that meter it": "—" if r == "waste" else (_other.get(r) or "none"),
} for r in RESOURCES])
print(COVERAGE.to_string(index=False))
print(f"Training: {IDX.min():%d %b %Y} → {TRAIN_END:%d %b %Y} · Monitoring: {EVAL_START:%d %b %Y} → {DATA_END:%d %b %Y}")

## 3 · Expected-value models (peer meters + calendar + weather)

In [ ]:
# ============================================================
# 3 · EXPECTED-VALUE MODELS (peer meters + calendar + weather), one per meter
# Each meter is predicted from the SAME-HOUR readings of the other meters of the same
# resource on campus (shared term dates, holidays, weather, campus-wide events), plus
# hour/day/season and temperature. A change confined to one building stands out.
# Out-of-sample errors: leave-one-calendar-quarter-out over the training period; they
# define "normal" for each meter and set its alert thresholds.
# ============================================================
def _calendar(ix, hourly):
    f = pd.DataFrame(index=ix)
    if hourly:
        f["hour"] = ix.hour
        f["temp"] = TEMP_H.reindex(ix).values
        f["temp_24h"] = TEMP_H.rolling(24, min_periods=1).mean().reindex(ix).values
    else:
        f["temp"] = TEMP_D.reindex(ix).values
    f["dow"] = ix.dayofweek
    f["doy_sin"] = np.sin(2 * np.pi * ix.dayofyear / 365.25)
    f["doy_cos"] = np.cos(2 * np.pi * ix.dayofyear / 365.25)
    return f


def model_features(res, b, ix):
    hourly = FREQ[res] == "h"
    f = _calendar(ix, hourly)
    for o in BLDS[res]:
        if o != b:
            f[f"peer_{o}"] = (METERS[res][o] if hourly else DAILY[(res, o)]).reindex(ix).values
    return f


def scale_stats(res, obs, exp):
    """Statistics alerts are judged on. Positive = adverse (more use, or less solar generation)."""
    r = SIGN[res] * (obs - exp)
    out = {}
    if FREQ[res] == "h":
        out["hour"] = r
        n = obs.resample("D").count()
        day = (r.resample("D").mean() * 24).where(n >= COMPLETENESS * 24)
        out["day"] = day
        if RESOURCES[res]["kind"] == "consumption":     # base load / night flow
            out["daymin"] = (obs.resample("D").min() - exp.resample("D").min()).where(n >= COMPLETENESS * 24)
    else:
        day = r
        out["day"] = day
    out["week"] = day.rolling(7, min_periods=7).sum()
    return out


SCALE_LABEL = {"hour": "1-hour", "day": "24-hour", "week": "7-day", "daymin": "daily-minimum (base load)"}
EXP, TH = {}, {}
t0 = time.time()
for res in RES_LIST:
    hourly = FREQ[res] == "h"
    ix_all = IDX if hourly else DAYS
    for b in BLDS[res]:
        y = METERS[res][b].reindex(ix_all)
        X = model_features(res, b, ix_all)
        tr = y.loc[:TRAIN_END].dropna()
        Xtr = X.loc[tr.index]
        quarters = tr.index.to_period("Q")
        r_oos = pd.Series(np.nan, index=tr.index)
        for q in quarters.unique():                     # leave-one-quarter-out
            m = np.asarray(quarters == q)
            mdl = lgb.LGBMRegressor(**LGB_PARAMS).fit(Xtr[~m], tr[~m])
            r_oos[m] = tr[m].values - mdl.predict(Xtr[m])
        final = lgb.LGBMRegressor(**LGB_PARAMS).fit(Xtr, tr)
        EXP[(res, b)] = pd.Series(final.predict(X), index=ix_all).clip(lower=0)
        obs_tr = y.loc[r_oos.index]
        null = scale_stats(res, obs_tr, obs_tr - r_oos)
        TH[(res, b)] = {k: {"hi": float(v.max()), "lo": float(v.min()), "n_null": int(v.notna().sum())}
                        for k, v in null.items()}
print(f"Trained {sum(len(BLDS[r]) for r in RES_LIST)} meter models in {time.time() - t0:.0f}s")

# ---------- out-of-sample accuracy on the monitoring period (real data never seen in training) ----------
FIT = []
for res in RES_LIST:
    for b in BLDS[res]:
        obs = METERS[res][b].loc[EVAL_START:]
        e = EXP[(res, b)].loc[obs.index]
        ok = obs.notna()
        level = METERS[res][b].loc[:TRAIN_END].mean()             # normalise by the meter's typical (training) level
        row = {"Resource": RESOURCES[res]["label"], "Building": b, "Use": BLD_USE.get(b, ""),
               "Peers in model": len(BLDS[res]) - 1,
               "Bias % of typical": round((e[ok] - obs[ok]).mean() / level * 100, 1)}
        if FREQ[res] == "h":
            row["Hourly error % of typical"] = round(np.sqrt(np.mean((obs[ok] - e[ok]) ** 2)) / level * 100, 1)
        od = DAILY[(res, b)].loc[EVAL_START:]
        ed = (EXP[(res, b)].resample("D").sum() if FREQ[res] == "h" else EXP[(res, b)]).loc[EVAL_START:]
        okd = od.notna()
        row["Daily error % of typical"] = round(np.sqrt(np.mean((od[okd] - ed[okd]) ** 2)) /
                                               DAILY[(res, b)].loc[:TRAIN_END].mean() * 100, 1)
        FIT.append(row)
FIT_DF = pd.DataFrame(FIT)
print(FIT_DF.groupby("Resource")[["Peers in model", "Bias % of typical", "Hourly error % of typical",
                                  "Daily error % of typical"]].median().round(1).to_string())

## 4 · Action verification engine (counterfactual + placebo tests in time and space)

In [ ]:
# ============================================================
# 4 · ACTION VERIFICATION — comparison-group counterfactual + placebo tests
#  1. Baseline model trained on the most recent 365 days of VALID data before the action
#     (outage days are skipped, not filled): day-of-week, temperature
#     and the same-day readings of COMPARISON meters (same resource, no logged action
#     during the whole period the model sees).
#  2. Predict what the meter would have read without the action (counterfactual).
#  3. Raw effect = favourable change vs. counterfactual over the measurement window
#     (less use for consumption resources; more generation for solar).
#  4. Placebos — cases where nothing happened, so any "effect" is noise:
#       • in TIME : the same meter at every eligible fake date before the real one
#       • in SPACE: each comparison meter, pretending IT got the action on the same date
#  5. Estimate = raw − mean time-placebo effect; 95% prediction interval from the robust
#     (MAD) spread of the pooled placebo deviations; permutation p-value from their ranks.
#  6. VERIFIED only if the 95% interval excludes zero AND p ≤ 5%. Otherwise NOT PROVEN.
#  Days with an alert on a meter are excluded (non-routine events, IPMVP practice).
# ============================================================
ACTION_REGISTRY = []
NONROUTINE_DAYS = {}
VERDICT_GOOD = {"consumption": "VERIFIED SAVING", "generation": "VERIFIED GAIN"}
VERDICT_BAD = {"consumption": "VERIFIED INCREASE", "generation": "VERIFIED LOSS"}


def comparison_buildings(res, b, T, post_days, registry=None):
    registry = ACTION_REGISTRY if registry is None else registry
    lo, hi = T - pd.Timedelta(days=2 * TRAIN_DAYS), T + pd.Timedelta(days=post_days)   # covers any 365 valid days
    busy = {a["building"] for a in registry if a["resource"] == res and lo <= pd.Timestamp(a["date"]) < hi}
    return [o for o in BLDS[res] if o != b and o not in busy]


def _design(res, controls, exclude_alert_days):
    X = pd.DataFrame(index=DAYS)
    X["dow"] = DAYS.dayofweek
    X["temp"] = TEMP_D.reindex(DAYS).values
    if controls:
        for o in controls:
            c = daily(res, o)
            if exclude_alert_days:
                bad = c.index.intersection(pd.DatetimeIndex(sorted(NONROUTINE_DAYS.get((res, o), set()))))
                c.loc[bad] = np.nan
            X[f"ctrl_{o}"] = c.values
    else:   # no untreated comparison meter → season + weather baseline
        X["doy_sin"] = np.sin(2 * np.pi * DAYS.dayofyear / 365.25)
        X["doy_cos"] = np.cos(2 * np.pi * DAYS.dayofyear / 365.25)
    return X


def _target(res, b, as_of, exclude_alert_days):
    y = daily(res, b)
    if as_of is not None:
        y = y.loc[:pd.Timestamp(as_of)]
    excluded = 0
    if exclude_alert_days:
        own = y.index.intersection(pd.DatetimeIndex(sorted(NONROUTINE_DAYS.get((res, b), set()))))
        excluded = len(own)
        y.loc[own] = np.nan
    return y, excluded


def _window_effect(res, y, X, T, post_days):
    tr = y.loc[: T - pd.Timedelta(days=1)].dropna().iloc[-TRAIN_DAYS:]     # most recent 365 VALID days (skips outages)
    te = y.loc[T: T + pd.Timedelta(days=post_days - 1)].dropna()
    if len(tr) < TRAIN_DAYS or len(te) < COMPLETENESS * post_days:
        return None
    mdl = lgb.LGBMRegressor(**LGB_SMALL).fit(X.loc[tr.index], tr.values)
    pred = pd.Series(mdl.predict(X.loc[te.index]), index=te.index).clip(lower=0)
    if pred.sum() <= 0:
        return None
    eff = SIGN[res] * (pred.sum() - te.sum()) / pred.sum() * 100     # favourable change, %
    return {"eff": float(eff), "pred": pred, "act": te}


def verify_action(res, b, T, post_days=42, registry=None, as_of=None, exclude_alert_days=True):
    T = pd.Timestamp(T).normalize()
    kind = RESOURCES[res]["kind"]
    y, excluded = _target(res, b, as_of, exclude_alert_days)
    valid = y.dropna()
    if valid.empty:
        return {"verdict": "INSUFFICIENT DATA", "why": "No valid readings for this meter."}
    start, end = valid.index.min(), valid.index.max()
    if T + pd.Timedelta(days=post_days - 1) > end:
        return {"verdict": "AWAITING DATA",
                "why": f"Needs {post_days} days of readings after the action; {max(0, (end - T).days + 1)} available."}
    controls = comparison_buildings(res, b, T, post_days, registry)
    X = _design(res, controls, exclude_alert_days)
    real = _window_effect(res, y, X, T, post_days)
    if real is None:
        return {"verdict": "INSUFFICIENT DATA",
                "why": f"Fewer than 365 valid days before the action, or fewer than 90% valid days in the {post_days}-day window."}
    null_time, Tk = [], T - pd.Timedelta(days=post_days)
    while y.loc[: Tk - pd.Timedelta(days=1)].notna().sum() >= TRAIN_DAYS:
        r = _window_effect(res, y, X, Tk, post_days)
        if r is not None:
            null_time.append(r["eff"])
        Tk -= pd.Timedelta(days=PLACEBO_STEP_DAYS)
    null_time = np.array(null_time)
    if len(null_time) < MIN_PLACEBOS:
        return {"verdict": "INSUFFICIENT HISTORY",
                "why": f"Only {len(null_time)} placebo dates fit before this action; at least {MIN_PLACEBOS} are needed "
                       f"for a 5% test (≈ 365 valid days + {post_days} + {MIN_PLACEBOS} days of history)."}
    null_space = []
    for o in controls:
        yo, _ = _target(res, o, as_of, exclude_alert_days)
        ro = _window_effect(res, yo, _design(res, [c for c in controls if c != o], exclude_alert_days), T, post_days)
        if ro is not None:
            null_space.append(ro["eff"])
    null_space = np.array(null_space)
    bias = float(null_time.mean())
    est = real["eff"] - bias
    pooled = np.concatenate([null_time - bias, null_space - null_space.mean() if len(null_space) > 1 else np.array([])])
    K = len(pooled)
    spread = 1.4826 * float(np.median(np.abs(pooled - np.median(pooled))))     # robust SD (MAD)
    half = float(stats.t.ppf(1 - ALPHA / 2, K - 1) * spread * np.sqrt(1 + 1 / K))
    lo, hi = est - half, est + half
    p = float((1 + np.sum(np.abs(pooled) >= abs(est))) / (K + 1))
    verdict = VERDICT_GOOD[kind] if (lo > 0 and p <= ALPHA) else (VERDICT_BAD[kind] if (hi < 0 and p <= ALPHA) else "NOT PROVEN")
    pre = y.loc[T - pd.Timedelta(days=post_days): T - pd.Timedelta(days=1)].mean()
    naive = float(SIGN[res] * (pre - real["act"].mean()) / pre * 100) if pre and pre > 0 else float("nan")
    act_total = float(real["act"].sum())
    cf_total = act_total / (1 - SIGN[res] * est / 100) if SIGN[res] * est < 100 else float(real["pred"].sum())
    scale = cf_total / float(real["pred"].sum())
    return {"verdict": verdict, "estimate_pct": est, "ci_lo_pct": lo, "ci_hi_pct": hi, "p_value": p,
            "raw_pct": real["eff"], "naive_pct": naive, "n_time": int(len(null_time)),
            "n_space": int(len(null_space)) if len(null_space) > 1 else 0, "null": pooled.tolist(),
            "post_days": post_days, "action_date": T, "controls": controls, "excluded_days": excluded,
            "actual_total": act_total, "counterfactual_total": cf_total,
            "benefit": cf_total * est / 100, "benefit_lo": cf_total * lo / 100, "benefit_hi": cf_total * hi / 100,
            "pred": real["pred"] * scale, "act": real["act"], "why": ""}


def explain_verdict(r, res):
    if "estimate_pct" not in r:
        return r["why"]
    label, unit, kind = RESOURCES[res]["label"], RESOURCES[res]["unit"], RESOURCES[res]["kind"]
    word = "saving" if kind == "consumption" else "gain"
    est, lo, hi, p = r["estimate_pct"], r["ci_lo_pct"], r["ci_hi_pct"], r["p_value"]
    ctrl = (f"{len(r['controls'])} comparison meter(s): {', '.join(r['controls'])}" if r["controls"]
            else "no untreated comparison meter available — season and weather baseline")
    base = (f"{label}: favourable change {est:+.1f}% versus the counterfactual ({ctrl}). 95% interval {lo:.1f}% to "
            f"{hi:.1f}%; permutation p = {p:.3f} against {r['n_time']} placebo dates for this meter and "
            f"{r['n_space']} comparison meters tested on the same date. ")
    if r["verdict"] == VERDICT_GOOD[kind]:
        return base + f"Both tests pass, so the {word} is counted at its lower bound: {r['benefit_lo']:,.0f} {unit} over {r['post_days']} days."
    if r["verdict"] == VERDICT_BAD[kind]:
        return base + "The change after this action was significantly UNFAVOURABLE."
    return base + f"At least one test fails, so the change is within ordinary variation. No {word} is counted."


print("Verification engine ready.")

## 5 · Alerts beyond each meter's own record, and meter-fault periods

In [ ]:
# ============================================================
# 5 · UNUSUAL-CHANGE ALERTS (monitoring period)
# Rule: a meter is flagged when its ADVERSE deviation from the expected value (more use;
# for solar, less generation) is LARGER than the largest out-of-sample deviation it showed
# in the whole training period, at any natural scale (1 hour, 24 hours, 7 days, and the
# daily minimum = base load for consumption meters). No tuned thresholds:
# each meter's own history sets its bar. If nothing changed, the chance that a given
# window beats the training record is about 1 / (number of training windows + 1).
# Sustained FAVOURABLE changes (7-day total beyond the training record on the good side)
# are listed separately: not problems, but nobody has explained them yet.
# ============================================================
SCALE_SPAN = {"hour": pd.Timedelta(hours=1), "day": pd.Timedelta(days=1),
              "week": pd.Timedelta(days=7), "daymin": pd.Timedelta(days=1)}


def _windows(stat, thr, scale, above=True):
    """Exceeding windows as (start, end, value, threshold) — window ends at the stat's timestamp period."""
    s = stat.loc[EVAL_START:]
    hit = s[s > thr] if above else s[s < thr]
    out = []
    for t, v in hit.items():
        if scale == "hour":
            out.append((t, t + SCALE_SPAN["hour"], v))
        elif scale == "week":
            out.append((t - pd.Timedelta(days=6), t + pd.Timedelta(days=1), v))
        else:
            out.append((t, t + pd.Timedelta(days=1), v))
    return out


def _merge(wins):
    """Union of overlapping windows → events, keeping every contributing test."""
    wins = sorted(wins, key=lambda w: w[0])
    ev = []
    for s, e, scale, v, thr in wins:
        if ev and s <= ev[-1]["end"]:                    # overlapping or touching windows = one event
            ev[-1]["end"] = max(ev[-1]["end"], e)
            ev[-1]["tests"].append((scale, v, thr, e))
        else:
            ev.append({"start": s, "end": e, "tests": [(scale, v, thr, e)]})
    return ev


def _fmt(v, res):
    return f"{v / 1000:,.1f} kL" if RESOURCES[res]["unit"] == "L" else f"{v:,.0f} {RESOURCES[res]['unit']}"


def describe(res, b, s, e, tests, obs, exp):
    """Plain description built only from measured quantities."""
    seg_o, seg_e = obs.loc[s: e - pd.Timedelta(seconds=1)], exp.loc[s: e - pd.Timedelta(seconds=1)]
    diff = SIGN[res] * (seg_o - seg_e)
    best = {}
    for scale, v, thr, _ in tests:                       # strongest exceedance per scale
        if scale not in best or v > best[scale][0]:
            best[scale] = (v, thr)
    parts = [f"{SCALE_LABEL[k]} deviation {_fmt(best[k][0], res)} vs. training record {_fmt(best[k][1], res)}"
             for k in SCALE_SPAN if k in best]
    peak = diff.idxmax() if diff.notna().any() else s
    peak_txt = f"largest hourly deviation at {peak:%a %d %b %H:%M}" if FREQ[res] == "h" else f"largest on {peak:%a %d %b}"
    return "; ".join(parts) + f"; {peak_txt}"


def detect_all(as_of=None):
    alerts, drops = [], []
    for res in RES_LIST:
        for b in BLDS[res]:
            obs, exp = METERS[res][b], EXP[(res, b)]
            st = scale_stats(res, obs, exp)
            th = TH[(res, b)]
            wins = []
            for scale, series in st.items():
                for s, e, v in _windows(series, th[scale]["hi"], scale, above=True):
                    wins.append((s, e, scale, v, th[scale]["hi"]))
            for ev in _merge(wins):
                s, e = ev["start"], ev["end"]
                seg_o, seg_e = obs.loc[s: e - pd.Timedelta(seconds=1)], exp.loc[s: e - pd.Timedelta(seconds=1)]
                excess = float((SIGN[res] * (seg_o - seg_e)).clip(lower=0).sum())
                alerts.append({"building": b, "resource": res, "start": s, "end": e,
                               "alarm_time": min(t[3] for t in ev["tests"]),
                               "tests": ", ".join(sorted({SCALE_LABEL[t[0]] for t in ev["tests"]})),
                               "excess": round(excess, 1),
                               "excess_pct": round(excess / max(float(seg_e.sum()), 1e-9) * 100, 1),
                               "description": describe(res, b, s, e, ev["tests"], obs, exp)})
            lo_w = [(s, e, "week", v, th["week"]["lo"]) for s, e, v in _windows(st["week"], th["week"]["lo"], "week", above=False)]
            for ev in _merge(lo_w):
                s, e = ev["start"], ev["end"]
                seg_o, seg_e = obs.loc[s: e - pd.Timedelta(seconds=1)], exp.loc[s: e - pd.Timedelta(seconds=1)]
                fav = float((SIGN[res] * (seg_e - seg_o)).sum())
                drops.append({"building": b, "resource": res, "start": s, "end": e,
                              "first_flagged": min(t[3] for t in ev["tests"]),
                              "favourable": round(fav, 1),
                              "favourable_pct": round(fav / max(float(seg_e.sum()), 1e-9) * 100, 1)})
    cols = ["building", "resource", "start", "end", "alarm_time", "tests", "excess", "excess_pct", "description"]
    A = pd.DataFrame(alerts, columns=cols).sort_values("alarm_time").reset_index(drop=True)
    A.insert(0, "alert_id", [f"AL-{i + 1:03d}" for i in range(len(A))])
    D = pd.DataFrame(drops, columns=["building", "resource", "start", "end", "first_flagged", "favourable", "favourable_pct"])
    D = D.sort_values("first_flagged").reset_index(drop=True)
    return A, D


ALERTS, DROPS = detect_all()
for _, a in ALERTS.iterrows():
    days = pd.date_range(a.start.normalize(), (a.end - pd.Timedelta(seconds=1)).normalize(), freq="D")
    NONROUTINE_DAYS.setdefault((a.resource, a.building), set()).update(days)

# expected number of alerts if nothing unusual happened (≈ windows / (training windows + 1))
EXPECTED_FALSE = 0.0
for res in RES_LIST:
    for b in BLDS[res]:
        st = scale_stats(res, METERS[res][b], EXP[(res, b)])
        for k, v in st.items():
            EXPECTED_FALSE += v.loc[EVAL_START:].notna().sum() / (TH[(res, b)][k]["n_null"] + 1)
print(f"Alerts in monitoring period: {len(ALERTS)} (if nothing had changed, about {EXPECTED_FALSE:.0f} window-level "
      f"exceedances would be expected by chance across all meters and scales)")
print(f"Unexplained sustained favourable changes: {len(DROPS)}")
print(ALERTS.groupby("resource").size().rename("alerts").to_string())


# ---------- meter data faults (days with <90% of hours, or a zero total, on a meter active in training) ----------
_f = []
for (res, b), days in FAULT_DAYS.items():
    ds = sorted(days)
    i = 0
    while i < len(ds):
        j = i
        while j + 1 < len(ds) and ds[j + 1] - ds[j] == pd.Timedelta(days=1):
            j += 1
        _f.append({"building": b, "resource": res, "first_day": ds[i], "last_day": ds[j], "days": j - i + 1})
        i = j + 1
METER_FAULTS = pd.DataFrame(_f, columns=["building", "resource", "first_day", "last_day", "days"]).sort_values("first_day")
for (res, b), days in FAULT_DAYS.items():
    NONROUTINE_DAYS.setdefault((res, b), set()).update(days)
print(f"Meter data faults: {len(METER_FAULTS)} periods, {int(METER_FAULTS.days.sum()) if len(METER_FAULTS) else 0} meter-days")

## 6 · Platform store: action register (starts empty), alerts, decisions

In [ ]:
# ============================================================
# 6 · PLATFORM STORE (SQLite) — action register (starts EMPTY), alert queue, decision log
# ============================================================
for _sfx in ("", "-wal", "-shm"):
    if os.path.exists(DB_PATH + _sfx):
        os.remove(DB_PATH + _sfx)


def db():
    c = sqlite3.connect(DB_PATH, timeout=30, check_same_thread=False)
    c.row_factory = sqlite3.Row
    return c


with db() as c:
    c.executescript("""
    CREATE TABLE actions(action_id TEXT PRIMARY KEY, title TEXT, building TEXT, resource TEXT, date TEXT,
        cost_inr REAL, claimed_pct REAL, notes TEXT, created_at TEXT);
    CREATE TABLE alerts(alert_id TEXT PRIMARY KEY, building TEXT, resource TEXT, start TEXT, end TEXT,
        alarm_time TEXT, tests TEXT, excess REAL, excess_pct REAL, description TEXT,
        status TEXT DEFAULT 'OPEN', note TEXT DEFAULT '', decided_at TEXT);
    CREATE TABLE decisions(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, alert_id TEXT, decision TEXT, note TEXT);
    """)
    for _, a in ALERTS.iterrows():
        c.execute("INSERT INTO alerts(alert_id,building,resource,start,end,alarm_time,tests,excess,excess_pct,description) "
                  "VALUES (?,?,?,?,?,?,?,?,?,?)",
                  (a.alert_id, a.building, a.resource, str(a.start), str(a.end), str(a.alarm_time), a.tests,
                   float(a.excess), float(a.excess_pct), a.description))


def load_actions():
    with db() as c:
        df = pd.read_sql("SELECT * FROM actions ORDER BY date", c)
    df["date"] = pd.to_datetime(df["date"])
    return df


def load_alerts():
    with db() as c:
        df = pd.read_sql("SELECT * FROM alerts", c)
    for col in ("start", "end", "alarm_time"):
        df[col] = pd.to_datetime(df[col])
    return df.sort_values("alarm_time").reset_index(drop=True)


MV_CACHE = {}


def sync_registry():
    ACTION_REGISTRY[:] = [dict(building=r.building, resource=r.resource, date=r.date) for r in load_actions().itertuples()]


def add_action(title, building, resource, date, cost_inr, claimed_pct, notes=""):
    date = pd.Timestamp(date).normalize()
    with db() as c:
        n = c.execute("SELECT COUNT(*) FROM actions").fetchone()[0] + 1
        while c.execute("SELECT 1 FROM actions WHERE action_id=?", (f"A{n}",)).fetchone():
            n += 1
        aid = f"A{n}"
        c.execute("INSERT INTO actions VALUES (?,?,?,?,?,?,?,?,?)",
                  (aid, title.strip(), building, resource, date.strftime("%Y-%m-%d"),
                   float(cost_inr) if cost_inr not in (None, "") else None,
                   float(claimed_pct) if claimed_pct not in (None, "") else None,
                   notes or "", datetime.now().isoformat(timespec="seconds")))
    sync_registry()
    MV_CACHE.clear()          # a new action can change other actions' comparison groups
    return aid


def decide_alert(alert_id, decision, note=""):
    ts = datetime.now().isoformat(timespec="seconds")
    with db() as c:
        c.execute("UPDATE alerts SET status=?, note=?, decided_at=? WHERE alert_id=?", (decision, note, ts, alert_id))
        c.execute("INSERT INTO decisions(ts, alert_id, decision, note) VALUES (?,?,?,?)", (ts, alert_id, decision, note))


def get_verification(action_id, post_days, as_of):
    a = load_actions().set_index("action_id").loc[action_id]
    as_of = pd.Timestamp(as_of).normalize()
    if as_of >= a.date + pd.Timedelta(days=int(post_days) - 1):
        as_of = DATA_END              # result depends only on data up to the window end
    key = (action_id, int(post_days), as_of)
    if key not in MV_CACHE:
        MV_CACHE[key] = verify_action(a.resource, a.building, a.date, int(post_days), as_of=as_of)
    return MV_CACHE[key]


sync_registry()
print(f"Store ready: {len(load_actions())} actions (the register starts empty — log real actions in the app), "
      f"{len(load_alerts())} alerts → {DB_PATH}")

## 7 · Verifier audit on real data (no action logged)

In [ ]:
# ============================================================
# 7 · VERIFIER AUDIT ON REAL DATA (no injected effects)
# Run the verifier on the first day of every month in the monitoring period, on every
# meter, with NO action logged. Compare with what a naive before/after comparison says.
# Real buildings do change, so a certified result here may be a genuine change — this
# audit shows how selective the verifier is relative to before/after, not a "truth" score.
# ============================================================
t0 = time.time()
AUDIT_POST = 42
from concurrent.futures import ThreadPoolExecutor
_tasks = [(res, b, T) for res in RES_LIST for b in BLDS[res]
          for T in pd.date_range(EVAL_START, DATA_END, freq="MS") if T + pd.Timedelta(days=AUDIT_POST - 1) <= DATA_END]


def _audit_one(task):
    res, b, T = task
    r = verify_action(res, b, T, AUDIT_POST, registry=[])
    row = {"Resource": RESOURCES[res]["label"], "Building": b, "Date": T.date(), "Verdict": r["verdict"]}
    if "estimate_pct" in r:
        row.update({"Naive before/after %": round(r["naive_pct"], 1), "Estimate %": round(r["estimate_pct"], 1),
                    "95% CI": f"{r['ci_lo_pct']:.1f} … {r['ci_hi_pct']:.1f}", "p": round(r["p_value"], 3)})
    return row


with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as _ex:     # LightGBM releases the GIL
    _rows = list(_ex.map(_audit_one, _tasks))
AUDIT_DF = pd.DataFrame(_rows)
_t = AUDIT_DF[AUDIT_DF["Verdict"].isin(list(VERDICT_GOOD.values()) + list(VERDICT_BAD.values()) + ["NOT PROVEN"])]
AUDIT_SUMMARY = {
    "tests": int(len(_t)),
    "naive_shows_benefit": int((_t["Naive before/after %"] > 0).sum()),
    "verified_benefit": int(_t["Verdict"].isin(VERDICT_GOOD.values()).sum()),
    "verified_adverse": int(_t["Verdict"].isin(VERDICT_BAD.values()).sum()),
    "not_testable": int(len(AUDIT_DF) - len(_t)),
}
print(f"Audit: {len(AUDIT_DF)} meter-dates in {time.time() - t0:.0f}s → {AUDIT_SUMMARY}")

## 8 · View helpers

In [ ]:
# ============================================================
# 8 · VIEW HELPERS (dashboard, ledger, report) — all quantities computed from real readings
# ============================================================
def fmt_qty(v, res):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "—"
    if RESOURCES[res]["unit"] == "L":
        return f"{v / 1000:,.1f} kL"
    return f"{v:,.0f} {RESOURCES[res]['unit']}"


def daily_expected(res, b):
    e = EXP[(res, b)]
    return e.resample("D").sum() if FREQ[res] == "h" else e


_first_sun = EVAL_START + pd.Timedelta(days=(6 - EVAL_START.dayofweek) % 7)
WEEK_ENDS = pd.date_range(_first_sun, DATA_END, freq="7D")
if WEEK_ENDS[-1] != DATA_END:
    WEEK_ENDS = WEEK_ENDS.append(pd.DatetimeIndex([DATA_END]))
ASOF_CHOICES = [(f"Week ending {d:%a %d %b %Y}", d.strftime("%Y-%m-%d")) for d in WEEK_ENDS]


def week_table(as_of):
    """Per meter: last-7-day actual vs expected, and whether the week beats the meter's training record."""
    as_of = pd.Timestamp(as_of)
    s = as_of - pd.Timedelta(days=6)
    rows = []
    for res in RES_LIST:
        for b in BLDS[res]:
            a = daily(res, b).loc[s:as_of]
            e = daily_expected(res, b).loc[s:as_of]
            ok = a.notna()
            act, exp = float(a[ok].sum()), float(e[ok].sum())
            dev = SIGN[res] * (act - exp)
            th = TH[(res, b)]["week"]
            status = ("beyond record (adverse)" if ok.all() and dev > th["hi"] else
                      "beyond record (favourable)" if ok.all() and dev < th["lo"] else
                      "incomplete week" if not ok.all() else "within normal range")
            rows.append({"res": res, "Resource": RESOURCES[res]["label"], "Building": b, "Use": BLD_USE.get(b, ""),
                         "actual": act, "expected": exp, "days": int(ok.sum()),
                         "dev_pct": (act - exp) / exp * 100 if exp > 0 else np.nan, "status": status})
    return pd.DataFrame(rows)


def visible_alerts(as_of):
    al = load_alerts()
    return al[al.alarm_time <= pd.Timestamp(as_of) + pd.Timedelta(days=1)]


def co2_of(res, qty):
    """kgCO2 for a quantity of a resource, only where a published factor applies."""
    if res == "electricity":
        return qty * EMISSION_FACTOR
    if res == "gas":
        return qty * GAS_EF
    if res == "solar":
        return qty * EMISSION_FACTOR          # generation displaces grid electricity
    return None


def ledger(as_of=None, post_days=42):
    as_of = pd.Timestamp(as_of or DATA_END)
    rows = []
    for a in load_actions().itertuples():
        r = get_verification(a.action_id, post_days, as_of) if a.date <= as_of else \
            {"verdict": "NOT STARTED", "why": "Action date is after the selected week."}
        kind = RESOURCES[a.resource]["kind"]
        has = "estimate_pct" in r
        counted = r["benefit_lo"] if has and r["verdict"] == VERDICT_GOOD[kind] else 0.0
        co2 = co2_of(a.resource, counted) if counted else None
        rows.append({
            "ID": a.action_id, "Action": a.title, "Building": a.building, "Resource": RESOURCES[a.resource]["label"],
            "Date": a.date.strftime("%Y-%m-%d"),
            "Claimed %": a.claimed_pct if a.claimed_pct is not None and not pd.isna(a.claimed_pct) else None,
            "Naive before/after %": round(r["naive_pct"], 1) if has else None,
            "Verdict": r["verdict"], "Estimate %": round(r["estimate_pct"], 1) if has else None,
            "95% CI": f"{r['ci_lo_pct']:.1f} … {r['ci_hi_pct']:.1f}" if has else "—",
            "p": round(r["p_value"], 3) if has else None,
            "Counted (lower bound)": fmt_qty(counted, a.resource) if counted else "—",
            "kgCO₂ (≥)": round(co2) if co2 else None,
            "_counted": counted, "_res": a.resource,
        })
    cols = ["ID", "Action", "Building", "Resource", "Date", "Claimed %", "Naive before/after %", "Verdict", "Estimate %",
            "95% CI", "p", "Counted (lower bound)", "kgCO₂ (≥)", "_counted", "_res"]
    return pd.DataFrame(rows, columns=cols)


def counted_totals(led):
    tot = {r: float(led.loc[led._res == r, "_counted"].sum()) for r in RES_LIST}
    tot["co2_kg"] = sum(co2_of(r, v) or 0 for r, v in tot.items() if r in RES_LIST)
    return tot


print("Views ready ·", len(ASOF_CHOICES), "weekly snapshots")

## 9 · Report and labelled real-data export

In [ ]:
# ============================================================
# 9 · EXPORTS — campus report (HTML / Markdown / JSON / CSV) and labelled REAL dataset
# Labelled dataset = real BDG2 readings + model expectation + the platform's own labels
# (alerts, meter faults, staff decisions, verification verdicts). Nothing is injected.
# ============================================================
import html as _html


def _inline(t):
    t = _html.escape(str(t))
    while t.count("**") >= 2:
        t = t.replace("**", "<b>", 1).replace("**", "</b>", 1)
    return t


def _md_to_html(lines):
    out, in_tab, first = [], False, False
    for ln in lines:
        if ln.startswith("|"):
            cells = [c.strip() for c in ln.strip("|").split("|")]
            if set("".join(cells)) <= set("-: "):
                continue
            if not in_tab:
                out.append("<table>"); in_tab, first = True, True
            tag = "th" if first else "td"
            first = False
            out.append("<tr>" + "".join(f"<{tag}>{_inline(c)}</{tag}>" for c in cells) + "</tr>")
            continue
        if in_tab:
            out.append("</table>"); in_tab = False
        if ln.startswith("# "):
            out.append(f"<h1>{_inline(ln[2:])}</h1>")
        elif ln.startswith("## "):
            out.append(f"<h2>{_inline(ln[3:])}</h2>")
        elif ln.startswith("- "):
            out.append(f"<li>{_inline(ln[2:])}</li>")
        elif ln.strip():
            out.append(f"<p>{_inline(ln)}</p>")
    if in_tab:
        out.append("</table>")
    return "\n".join(out)


def build_report(as_of=None, post_days=42):
    as_of = pd.Timestamp(as_of or DATA_END)
    wk, led, al = week_table(as_of), ledger(as_of, post_days), visible_alerts(as_of)
    tot = counted_totals(led)
    L = ["# GreenLedger — Campus Resource Report",
         f"BDG2 campus **{SITE}** · week ending {as_of:%d %b %Y} · measurement window {post_days} days · "
         f"generated {datetime.now():%d %b %Y %H:%M}", "",
         "## 1. Resources covered", "", "| Resource | Status | Other BDG2 campuses that meter it |", "|---|---|---|"]
    L += [f"| {r['Resource']} | {r['Status on this campus']} | {r['Other BDG2 campuses that meter it']} |" for _, r in COVERAGE.iterrows()]
    L += ["", "## 2. This week vs expected", "",
          "| Resource | Meters | Actual | Expected | Difference | Meters beyond their training record |", "|---|---|---|---|---|---|"]
    for res in RES_LIST:
        w = wk[wk.res == res]
        a, e = w.actual.sum(), w.expected.sum()
        beyond = int(w.status.str.startswith("beyond").sum())
        L.append(f"| {RESOURCES[res]['label']} | {len(w)} | {fmt_qty(a, res)} | {fmt_qty(e, res)} | "
                 f"{(a - e) / e * 100 if e else 0:+.1f}% | {beyond} |")
    L += ["", "## 3. Alerts (adverse changes beyond each meter's own training record)", "",
          f"{len(al)} alerts up to this week · open {int((al.status == 'OPEN').sum())} · confirmed "
          f"{int((al.status == 'CONFIRMED').sum())} · dismissed {int((al.status == 'DISMISSED').sum())}", "",
          "| Alert | Raised | Meter | Excess | Evidence |", "|---|---|---|---|---|"]
    for e in al.sort_values("alarm_time", ascending=False).head(15).itertuples():
        L.append(f"| {e.alert_id} | {e.alarm_time:%d %b %H:%M} | {e.building} · {RESOURCES[e.resource]['label']} | "
                 f"{fmt_qty(e.excess, e.resource)} (+{e.excess_pct:.0f}%) | {e.description} |")
    mf = METER_FAULTS[METER_FAULTS.first_day <= as_of]
    L += ["", f"## 4. Meter data faults: {len(mf)} periods, {int(mf.days.sum()) if len(mf) else 0} meter-days excluded", ""]
    L += ["## 5. Logged actions — verified against a counterfactual", ""]
    if len(led):
        L += ["| ID | Action | Meter | Claimed | Naive | Verdict | Estimate (95% CI) | p | Counted |", "|---|---|---|---|---|---|---|---|---|"]
        for _, r in led.iterrows():
            L.append(f"| {r['ID']} | {r['Action']} | {r['Building']} · {r['Resource']} | "
                     f"{'' if r['Claimed %'] is None or pd.isna(r['Claimed %']) else str(r['Claimed %']) + '%'} | "
                     f"{'' if r['Naive before/after %'] is None or pd.isna(r['Naive before/after %']) else str(r['Naive before/after %']) + '%'} | "
                     f"{r['Verdict']} | {'' if r['Estimate %'] is None or pd.isna(r['Estimate %']) else str(r['Estimate %']) + '%'} ({r['95% CI']}) | "
                     f"{'' if r['p'] is None or pd.isna(r['p']) else r['p']} | {r['Counted (lower bound)']} |")
        L += ["", "Counted totals (95% lower bounds of verified results only): " +
              " · ".join(fmt_qty(tot[r], r) + " " + RESOURCES[r]["label"].lower() for r in RES_LIST if tot[r] > 0) +
              (f" · ≥ {tot['co2_kg'] / 1000:,.2f} tCO₂" if tot["co2_kg"] > 0 else "")]
    else:
        L += ["No actions logged yet. Savings are never assumed: an action must be logged and verified to count."]
    L += ["", "## 6. Method", "",
          "- Expected values: LightGBM (library defaults) per meter using same-hour readings of peer meters, calendar and weather.",
          "- Alerts: deviation larger than the meter's own largest out-of-sample deviation in the training period (1 h, 24 h, 7 d, daily minimum).",
          "- Verification: counterfactual from comparison meters; placebo tests in time and space; verified only if the 95% interval excludes zero and p ≤ 5%.",
          f"- CO₂: electricity and solar at {EF_SOURCE}; natural gas at {GAS_EF_SOURCE}. No factor is assumed for chilled water, hot water or steam.",
          "- All readings are real Building Data Genome 2 meter data; no values are generated, injected or interpolated.", ""]
    md = "\n".join(L)
    html_doc = ("<!doctype html><html><head><meta charset='utf-8'><title>GreenLedger report</title><style>"
                "body{font-family:Inter,Segoe UI,Arial,sans-serif;max-width:1000px;margin:32px auto;color:#14202e;line-height:1.5;padding:0 18px}"
                "h1{font-size:26px}h2{font-size:18px;margin-top:26px;border-bottom:2px solid #10B981;padding-bottom:4px}"
                "table{border-collapse:collapse;width:100%;font-size:12.5px;margin:10px 0}th,td{border:1px solid #d5dde6;padding:6px 8px;text-align:left;vertical-align:top}"
                "th{background:#eef3f7}</style></head><body>" + _md_to_html(L) + "</body></html>")
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    paths = {k: os.path.join(OUT_DIR, f"greenledger_report_{stamp}.{k}") for k in ("html", "md", "json")}
    paths["csv"] = os.path.join(OUT_DIR, f"greenledger_ledger_{stamp}.csv")
    open(paths["md"], "w", encoding="utf-8").write(md)
    open(paths["html"], "w", encoding="utf-8").write(html_doc)
    led.drop(columns=["_counted", "_res"]).to_csv(paths["csv"], index=False)
    json.dump({"site": SITE, "as_of": str(as_of.date()), "post_days": post_days,
               "counted_totals": {k: v for k, v in tot.items()}, "ledger": led.drop(columns=["_counted", "_res"]).to_dict("records"),
               "alerts": int(len(al)), "meter_fault_periods": int(len(mf)), "audit": AUDIT_SUMMARY},
              open(paths["json"], "w"), indent=2, default=str)
    return md, paths


def export_labelled_dataset():
    """Real readings + expectations + platform labels, as CSV files in one ZIP."""
    path = os.path.join(OUT_DIR, f"greenledger_{SITE}_labelled_real_data.zip")
    al = load_alerts()
    with db() as c:
        dec = pd.read_sql("SELECT * FROM decisions", c)
    frames = []
    for res in RES_LIST:
        for b in BLDS[res]:
            obs = METERS[res][b]
            lab = pd.Series("", index=obs.index)
            for e in al[(al.resource == res) & (al.building == b)].itertuples():
                lab[(lab.index >= e.start) & (lab.index < e.end)] = e.alert_id
            fault = pd.Series(obs.index.normalize().isin(pd.DatetimeIndex(sorted(FAULT_DAYS.get((res, b), set())))),
                              index=obs.index)
            frames.append(pd.DataFrame({"timestamp": obs.index, "resource": res, "building": b,
                                        "reading": obs.values, "expected": EXP[(res, b)].reindex(obs.index).values,
                                        "alert_id": lab.values, "meter_fault_day": fault.values}))
    readme = (f"GreenLedger labelled dataset — REAL data only\nCampus: Building Data Genome 2 site '{SITE}'.\n"
              "readings.csv: hourly meter readings exactly as published in BDG2 (fault days blanked), the model's expected "
              "value, the alert covering that hour (if any) and a meter-fault flag.\n"
              "alerts.csv: alerts with staff status/notes; decisions.csv: decision log; meter_faults.csv; "
              "sustained_changes.csv; actions.csv: logged actions with verdicts; parameters.csv.\n"
              "No values were generated, injected or interpolated.\n")
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("README.txt", readme)
        z.writestr("readings.csv", pd.concat(frames).to_csv(index=False))
        z.writestr("alerts.csv", al.to_csv(index=False))
        z.writestr("decisions.csv", dec.to_csv(index=False))
        z.writestr("meter_faults.csv", METER_FAULTS.to_csv(index=False))
        z.writestr("sustained_changes.csv", DROPS.to_csv(index=False))
        z.writestr("actions.csv", ledger(DATA_END).drop(columns=["_counted", "_res"]).to_csv(index=False))
        z.writestr("parameters.csv", PARAMETER_TABLE.to_csv(index=False))
        z.writestr("buildings_and_coverage.csv", COVERAGE.to_csv(index=False))
    return path


_md, _paths = build_report()
print("Report test OK:", {k: os.path.basename(v) for k, v in _paths.items()})

## 10 · Interim app (Gradio)

In [ ]:
# ============================================================
# 10 · GREENLEDGER APP (Gradio 4 / 5 / 6 compatible)
# ============================================================
import gradio as gr
from plotly.subplots import make_subplots

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@500;700&family=Space+Grotesk:wght@600;700&display=swap');
:root{--bg:#0B1220;--s1:#111A2C;--s2:#182238;--s3:#1F2A44;--bd:#2A3A5C;--tx:#E6EDF7;--tx2:#B6C2D6;--mu:#8794AB;
--em:#10B981;--em2:#34D399;--am:#F59E0B;--am2:#FBBF24;--rd:#EF4444;--rd2:#F87171;--bl:#3B82F6;--bl2:#60A5FA;}
html,body,gradio-app,.gradio-container{background:var(--bg)!important;color:var(--tx)!important;font-family:Inter,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif!important}
.gradio-container{max-width:1500px!important;margin:0 auto!important}
.gradio-container *{--body-background-fill:var(--bg);--background-fill-primary:var(--s1);--background-fill-secondary:var(--s2);
--block-background-fill:var(--s1);--block-border-color:var(--bd);--border-color-primary:var(--bd);--input-background-fill:var(--s2);
--body-text-color:var(--tx);--block-label-text-color:var(--tx2);--block-title-text-color:var(--tx2);--color-accent:var(--em);
--table-even-background-fill:var(--s1);--checkbox-label-background-fill:var(--s2);
--checkbox-label-background-fill-selected:rgba(16,185,129,.18);--checkbox-label-text-color:var(--tx);
--checkbox-label-text-color-selected:#fff;--checkbox-label-border-color:var(--bd);--block-label-background-fill:var(--s2);
--block-label-text-color:var(--tx2);--block-label-border-color:var(--bd);--input-placeholder-color:var(--mu);--table-odd-background-fill:var(--s2);--table-row-focus:var(--s3);--neutral-50:var(--s1);
--neutral-100:var(--s2);--neutral-200:var(--bd);--neutral-700:var(--tx);--neutral-800:var(--tx);--neutral-900:#fff}
.gl-head{display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap;padding:22px 28px;margin:14px 0 10px;
background:linear-gradient(120deg,#111A2C,#16304a);border:1px solid var(--bd);border-radius:14px}
.gl-brand{display:flex;gap:14px;align-items:center}.gl-logo{width:50px;height:50px;border-radius:12px;background:linear-gradient(135deg,#10B981,#047857);
display:flex;align-items:center;justify-content:center;font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff}
.gl-title{font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff;letter-spacing:-.02em}.gl-sub{font:500 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);margin-top:4px;letter-spacing:.04em}
.gl-pill{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;padding:8px 12px;border-radius:8px;border:1px solid rgba(16,185,129,.45);background:rgba(16,185,129,.12);color:var(--em2)}
.gl-intro{background:var(--s1);border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:10px;padding:14px 18px;margin:12px 0;font-size:14px;color:var(--tx2);line-height:1.6}
.gl-intro b{color:#fff}
.gl-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:12px;margin:12px 0}
.gl-card{background:linear-gradient(160deg,var(--s1),var(--s2));border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:12px;padding:14px 16px}
.gl-card.warn{border-left-color:var(--am)}.gl-card.bad{border-left-color:var(--rd)}.gl-card.info{border-left-color:var(--bl)}.gl-card.mute{border-left-color:#5E6B85}
.gl-v{font:700 26px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:#fff;letter-spacing:-.02em}.gl-u{font-size:13px;color:var(--tx2);margin-left:4px}
.gl-l{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);text-transform:uppercase;letter-spacing:.08em;margin-top:8px}
.gl-h{font-size:12px;color:var(--mu);margin-top:4px;line-height:1.45}
.gl-verdict{border-radius:14px;padding:20px 24px;margin:10px 0;border:1px solid var(--bd)}
.gl-verdict .big{font:700 30px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:-.01em}.gl-verdict .num{font:700 20px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-top:6px;color:#fff}
.gl-verdict p{color:var(--tx2);font-size:14px;line-height:1.6;margin:10px 0 0}
.gl-ok{background:rgba(16,185,129,.12);border-color:rgba(16,185,129,.5)}.gl-ok .big{color:var(--em2)}
.gl-no{background:rgba(245,158,11,.10);border-color:rgba(245,158,11,.45)}.gl-no .big{color:var(--am2)}
.gl-bad{background:rgba(239,68,68,.10);border-color:rgba(239,68,68,.45)}.gl-bad .big{color:var(--rd2)}
.gl-wait{background:var(--s2)}.gl-wait .big{color:var(--tx2)}
table.gl-t{width:100%;border-collapse:collapse;font:13px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin:8px 0}
table.gl-t th{background:var(--s3);color:#fff;text-align:left;padding:9px 10px;font-size:11px;text-transform:uppercase;letter-spacing:.06em}
table.gl-t td{padding:8px 10px;border-bottom:1px solid var(--bd);color:var(--tx)}
.z-hi{background:rgba(239,68,68,.22)!important;color:#FECACA!important}.z-lo{background:rgba(16,185,129,.20)!important;color:#A7F3D0!important}
.gl-sec{font:700 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);letter-spacing:.12em;text-transform:uppercase;margin:22px 0 8px}
.gl-note{font-size:12.5px;color:var(--mu);font-style:italic;margin:6px 0}
.gl-msg{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--em2);padding:8px 12px;background:rgba(16,185,129,.1);border:1px solid rgba(16,185,129,.35);border-radius:8px;display:inline-block}
.gl-msg.warn{color:var(--am2);background:rgba(245,158,11,.1);border-color:rgba(245,158,11,.4)}
.tabs button{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace!important;letter-spacing:.06em!important;text-transform:uppercase!important}
.tabs button.selected{color:var(--em2)!important;border-color:var(--em2)!important}
button.primary{background:linear-gradient(135deg,#10B981,#059669)!important;color:#fff!important;border:none!important;font-weight:700!important}
button.secondary{background:var(--s2)!important;color:var(--tx)!important;border:1px solid var(--bd)!important;font-weight:600!important}
button.secondary:hover{border-color:var(--em2)!important;color:var(--em2)!important}
.gl-foot{margin:36px 0 18px;padding-top:14px;border-top:1px solid var(--bd);display:flex;justify-content:space-between;flex-wrap:wrap;gap:10px;
font:500 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--mu);letter-spacing:.06em;text-transform:uppercase}
</style>
"""

PL = dict(plot_bgcolor="#111A2C", paper_bgcolor="#111A2C",
          font=dict(family="Inter, sans-serif", size=12, color="#E6EDF7"),
          margin=dict(t=56, b=44, l=60, r=24),
          legend=dict(orientation="h", y=-0.18, bgcolor="rgba(0,0,0,0)"),
          hoverlabel=dict(bgcolor="#1F2A44", font=dict(color="#fff")))
GRID = dict(gridcolor="#243352", zerolinecolor="#324369", linecolor="#324369")


def _fig(fig, title, h=380):
    fig.update_layout(title=dict(text=title, font=dict(size=15, color="#fff")), height=h, **PL)
    fig.update_xaxes(**GRID); fig.update_yaxes(**GRID)
    return fig



def card(v, label, help_="", cls="", unit=""):
    u = f'<span class="gl-u">{unit}</span>' if unit else ""
    h = f'<div class="gl-h">{help_}</div>' if help_ else ""
    return f'<div class="gl-card {cls}"><div class="gl-v">{v}{u}</div><div class="gl-l">{label}</div>{h}</div>'


def intro(text):
    return f'<div class="gl-intro">{text}</div>'


def msg(text, warn=False):
    return f'<div class="gl-msg{" warn" if warn else ""}">{text}</div>'


def _asof(v):
    return pd.Timestamp(v or ASOF_CHOICES[-1][1])


def _tbl_html(df):
    return df.to_html(index=False, classes="gl-t", border=0, escape=True)


# ---------------- OVERVIEW ----------------
def ui_overview(as_of_v, window):
    as_of = _asof(as_of_v)
    wk, al = week_table(as_of), visible_alerts(as_of)
    led = ledger(as_of, int(window))
    tot = counted_totals(led)
    cards = []
    for res in RES_LIST:
        w = wk[wk.res == res]
        a, e = w.actual.sum(), w.expected.sum()
        dev = (a - e) / e * 100 if e else 0
        n_adv = int((w.status == "beyond record (adverse)").sum())
        cards.append(card(fmt_qty(a, res), f"{RESOURCES[res]['label']} · this week",
                          f"{dev:+.1f}% vs expected · {len(w)} meters · {n_adv} beyond record",
                          "bad" if n_adv else ""))
    co2 = sum(co2_of(r, wk[wk.res == r].actual.sum()) or 0 for r in ("electricity", "gas") if r in RES_LIST)
    if "solar" in RES_LIST:
        co2 -= co2_of("solar", wk[wk.res == "solar"].actual.sum())
    cards.append(card(f"{co2 / 1000:,.1f}", "Net CO₂ this week", "electricity + gas − solar (published factors only)", "mute", "t"))
    cards.append(card(str(int((al.status == "OPEN").sum())), "Open alerts", f"{len(al)} raised so far", "warn"))
    cards.append(card(f"{tot['co2_kg'] / 1000:,.2f}" if tot["co2_kg"] else "0", "Verified CO₂ reduction",
                      "lower bound of verified actions only", "", "t"))
    cards_html = '<div class="gl-cards">' + "".join(cards) + "</div>"

    fig = make_subplots(rows=len(RES_LIST), cols=1, shared_xaxes=True, vertical_spacing=0.045,
                        subplot_titles=[f"{RESOURCES[r]['label']} — campus total per day "
                                        f"({'kL' if RESOURCES[r]['unit'] == 'L' else RESOURCES[r]['unit']})" for r in RES_LIST])
    s = EVAL_START - pd.Timedelta(days=28)
    for i, res in enumerate(RES_LIST, 1):
        k = 1000 if RESOURCES[res]["unit"] == "L" else 1
        A = pd.concat([daily(res, b).loc[s:as_of] for b in BLDS[res]], axis=1)
        E = pd.concat([daily_expected(res, b).loc[s:as_of] for b in BLDS[res]], axis=1)
        mask = A.notna()
        A_s, E_s = A.sum(axis=1, min_count=1) / k, E.where(mask).sum(axis=1, min_count=1) / k
        fig.add_trace(go.Scatter(x=E_s.index, y=E_s, name="Expected", line=dict(color="#8794AB", dash="dot", width=2),
                                 showlegend=(i == 1)), row=i, col=1)
        fig.add_trace(go.Scatter(x=A_s.index, y=A_s, name="Actual", line=dict(color=RESOURCES[res]["color"], width=2.3),
                                 showlegend=(i == 1)), row=i, col=1)
        ar = al[al.resource == res]
        if len(ar):
            d = ar.alarm_time.dt.normalize()
            fig.add_trace(go.Scatter(x=d, y=[A_s.reindex([x]).fillna(0).iloc[0] for x in d], mode="markers",
                                     marker=dict(symbol="triangle-up", size=11, color="#F87171", line=dict(color="#fff", width=1)),
                                     name="Alert", showlegend=(i == 1), text=ar.alert_id + " · " + ar.building,
                                     hovertemplate="%{text}<extra></extra>"), row=i, col=1)
        for a in load_actions().itertuples():
            if a.resource == res and s <= a.date <= as_of:
                fig.add_vline(x=a.date, line=dict(color="#34D399", width=1, dash="dash"), row=i, col=1)
        fig.add_vline(x=EVAL_START, line=dict(color="#5E6B85", width=1), row=i, col=1)
    _fig(fig, "Campus resources (grey line = start of monitoring · green dashed = logged action · red ▲ = alert)",
         220 * len(RES_LIST) + 80)
    for ann in fig.layout.annotations:
        ann.font = dict(size=12, color="#B6C2D6")

    rows = []
    for r in wk.itertuples():
        cls = "z-hi" if r.status == "beyond record (adverse)" else ("z-lo" if r.status == "beyond record (favourable)" else "")
        dev = "—" if pd.isna(r.dev_pct) else f"{r.dev_pct:+.0f}%"
        rows.append(f"<tr><td>{r.Resource}</td><td><b>{r.Building}</b></td><td>{r.Use}</td>"
                    f"<td class='{cls}'>{fmt_qty(r.actual, r.res)} ({dev})</td><td>{fmt_qty(r.expected, r.res)}</td>"
                    f"<td>{r.days}/7</td><td class='{cls}'>{r.status}</td></tr>")
    score = ('<table class="gl-t"><tr><th>Resource</th><th>Building</th><th>Use</th><th>Actual (vs expected)</th>'
             '<th>Expected</th><th>Valid days</th><th>Status</th></tr>' + "".join(rows) + "</table>"
             '<div class="gl-note">Coloured only when the week is beyond the meter\'s own 18-month out-of-sample record '
             '(red = adverse, green = favourable). For solar, adverse = less generation.</div>')
    return cards_html, fig, score


# ---------------- ALERTS ----------------
def _alerts_view(as_of_v, res_f, status_f):
    al = visible_alerts(_asof(as_of_v))
    if res_f and res_f != "All":
        al = al[al.resource == res_f]
    if status_f and status_f != "All":
        al = al[al.status == status_f]
    al = al.sort_values("alarm_time", ascending=False)
    tbl = pd.DataFrame({"Alert": al.alert_id, "Raised": al.alarm_time.dt.strftime("%d %b %H:%M"),
                        "Building": al.building, "Resource": al.resource.map(lambda r: RESOURCES[r]["label"]),
                        "Tests beyond record": al.tests,
                        "Excess": [fmt_qty(e, r) for e, r in zip(al.excess, al.resource)],
                        "Excess %": al.excess_pct.round(0), "Status": al.status})
    choices = [(f"{r.alert_id} · {r.building} · {RESOURCES[r.resource]['label']} · {r.alarm_time:%d %b}", r.alert_id)
               for r in al.itertuples()]
    return tbl, choices


def ui_alerts(as_of_v, res_f="All", status_f="All"):
    tbl, choices = _alerts_view(as_of_v, res_f, status_f)
    first = choices[0][1] if choices else None
    fig, expl = ui_alert_detail(first)
    return tbl, gr.update(choices=choices, value=first), fig, expl


def ui_alert_detail(alert_id):
    if not alert_id:
        return _fig(go.Figure(), "No alert selected"), intro("No alerts for this selection.")
    e = load_alerts().set_index("alert_id").loc[alert_id]
    res, b = e.resource, e.building
    hourly = FREQ[res] == "h"
    s, t = (e.start - pd.Timedelta(days=4), e.end + pd.Timedelta(days=2)) if hourly else \
           (e.start - pd.Timedelta(days=21), e.end + pd.Timedelta(days=14))
    obs, exp = METERS[res][b].loc[s:t], EXP[(res, b)].loc[s:t]
    key = "hour" if hourly else "day"
    bound = exp + SIGN[res] * TH[(res, b)][key]["hi"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=bound.index, y=bound.clip(lower=0), name="Expected ± training record (1-step)",
                             line=dict(color="#F87171", dash="dot", width=1)))
    fig.add_trace(go.Scatter(x=exp.index, y=exp, name="Expected", line=dict(color="#8794AB", dash="dash")))
    fig.add_trace(go.Scatter(x=obs.index, y=obs, name="Actual", line=dict(color=RESOURCES[res]["color"], width=2.4)))
    fig.add_vrect(x0=e.start, x1=e.end, fillcolor="rgba(239,68,68,0.15)", line_width=0)
    fig.add_vline(x=e.alarm_time, line=dict(color="#F87171", dash="dash"))
    unit = RESOURCES[res]["unit"] + ("/h" if hourly else "/day")
    _fig(fig, f"{alert_id} · {b} · {RESOURCES[res]['label']} ({unit}) — shaded = flagged period", 400)
    dur = (e.end - e.start) / pd.Timedelta(hours=1)
    dur_txt = f"{dur:.0f} hours" if dur < 48 else f"{dur / 24:.1f} days"
    co2 = co2_of(res, e.excess)
    co2_txt = f" ≈ {co2:,.0f} kgCO₂." if co2 else "."
    rupee = f" ≈ ₹{e.excess * TARIFF_INR_PER_KWH:,.0f} at your tariff." if TARIFF_INR_PER_KWH and res == "electricity" else ""
    what = "less generation than expected" if RESOURCES[res]["kind"] == "generation" else "more than expected"
    html_ = intro(f"<b>{b} · {RESOURCES[res]['label']} ({BLD_USE.get(b, '')})</b>: {fmt_qty(e.excess, res)} {what} "
                  f"(+{e.excess_pct:.0f}%) over {dur_txt}, {e.start:%a %d %b %H:%M} → {e.end:%a %d %b %H:%M}{co2_txt}{rupee}<br>"
                  f"<b>Evidence:</b> {e.description}.<br><b>Raised:</b> {e.alarm_time:%a %d %b %H:%M} · "
                  f"<b>Status:</b> {e.status}{(' — ' + e.note) if e.note else ''}")
    return fig, html_


def _decision_log():
    with db() as c:
        return pd.read_sql("SELECT ts AS 'When', alert_id AS 'Alert', decision AS 'Decision', note AS 'Note' "
                           "FROM decisions ORDER BY id DESC LIMIT 30", c)


def ui_decide(alert_id, decision, note, as_of_v, res_f, status_f):
    if not alert_id:
        t, dd, f, x = ui_alerts(as_of_v, res_f, status_f)
        return msg("Pick an alert first.", True), t, dd, f, x, _decision_log()
    decide_alert(alert_id, decision, note or "")
    t, _, _, _ = ui_alerts(as_of_v, res_f, status_f)
    f, x = ui_alert_detail(alert_id)
    return msg(f"{alert_id} → {decision}"), t, gr.update(value=alert_id), f, x, _decision_log()


def ui_faults(as_of_v):
    mf = METER_FAULTS[METER_FAULTS.first_day <= _asof(as_of_v)].copy()
    mf["resource"] = mf.resource.map(lambda r: RESOURCES[r]["label"])
    mf["first_day"] = mf.first_day.dt.strftime("%d %b %Y"); mf["last_day"] = mf.last_day.dt.strftime("%d %b %Y")
    mf.columns = ["Building", "Resource", "First day", "Last day", "Days"]
    dr = DROPS[DROPS.first_flagged <= _asof(as_of_v) + pd.Timedelta(days=1)].copy()
    out_dr = pd.DataFrame({"Building": dr.building, "Resource": dr.resource.map(lambda r: RESOURCES[r]["label"]),
                           "From": dr.start.dt.strftime("%d %b"), "To": dr.end.dt.strftime("%d %b"),
                           "Favourable change": [fmt_qty(v, r) for v, r in zip(dr.favourable, dr.resource)],
                           "%": dr.favourable_pct})
    return mf.sort_values("Days", ascending=False), out_dr


# ---------------- ACTION REGISTER ----------------
def _ledger_view(as_of_v, window):
    led = ledger(_asof(as_of_v), int(window))
    return led.drop(columns=["_counted", "_res"])


def _action_choices():
    return [(f"{a.action_id} · {a.title} · {a.building} · {RESOURCES[a.resource]['label']}", a.action_id)
            for a in load_actions().itertuples()]


def ui_res_buildings(res):
    return gr.update(choices=BLDS.get(res, []), value=(BLDS.get(res) or [None])[0])


def ui_log_action(title, res, building, date, cost, claimed, notes, as_of_v, window):
    try:
        if not title or not str(title).strip():
            raise ValueError("Describe the action.")
        if res not in RES_LIST or building not in BLDS[res]:
            raise ValueError("Choose a resource and one of its metered buildings.")
        d = pd.Timestamp(str(date).strip())
        if d < IDX.min() or d > DATA_END:
            raise ValueError(f"Date must be between {IDX.min():%Y-%m-%d} and {DATA_END:%Y-%m-%d} (meter data range).")
        aid = add_action(title, building, res, d, cost, claimed, notes)
        m = msg(f"Logged {aid}. Open 'Verify' to test it.")
    except Exception as ex:
        m = msg(f"Not logged: {ex}", True)
    ch = _action_choices()
    return m, _ledger_view(as_of_v, window), gr.update(choices=ch, value=ch[-1][1] if ch else None)


# ---------------- VERIFY ----------------
def ui_verify(action_id, as_of_v, window):
    empty = _fig(go.Figure(), "")
    if not action_id:
        return intro("No action selected. Log a real action in the <b>Action Register</b> first — "
                     "GreenLedger does not ship with pre-made actions."), empty, empty, ""
    a = load_actions().set_index("action_id").loc[action_id]
    as_of = _asof(as_of_v)
    r = get_verification(action_id, int(window), as_of) if a.date <= as_of else \
        {"verdict": "NOT STARTED", "why": f"This action is dated {a.date:%d %b %Y}, after the selected week."}
    kind = RESOURCES[a.resource]["kind"]
    if "estimate_pct" not in r:
        return f'<div class="gl-verdict gl-wait"><div class="big">{r["verdict"]}</div><p>{r["why"]}</p></div>', empty, empty, ""
    est, lo, hi = r["estimate_pct"], r["ci_lo_pct"], r["ci_hi_pct"]
    cls = "gl-ok" if r["verdict"] == VERDICT_GOOD[kind] else ("gl-bad" if r["verdict"] == VERDICT_BAD[kind] else "gl-no")
    extra = ""
    if r["verdict"] == VERDICT_GOOD[kind]:
        extra = f" · counted: ≥ {fmt_qty(r['benefit_lo'], a.resource)}"
        c = co2_of(a.resource, r["benefit_lo"])
        if c:
            extra += f" · ≥ {c / 1000:,.2f} tCO₂"
    card_ = (f'<div class="gl-verdict {cls}"><div class="big">{r["verdict"]}</div>'
             f'<div class="num">{est:+.1f}%  (95% CI {lo:.1f}% … {hi:.1f}%, p = {r["p_value"]:.3f}){extra}</div>'
             f'<p>{explain_verdict(r, a.resource)}</p></div>')
    claimed = "—" if a.claimed_pct is None or pd.isna(a.claimed_pct) else f"{a.claimed_pct:.0f}%"
    comp = ('<div class="gl-cards">'
            + card(claimed, "Claimed", "as entered when the action was logged", "mute")
            + card(f"{r['naive_pct']:.1f}%", "Naive before/after", f"{r['post_days']} days before vs after — no counterfactual", "warn")
            + card(f"{est:.1f}%", "GreenLedger estimate", f"95% CI {lo:.1f} … {hi:.1f}%", "")
            + card(f"{r['n_time']} + {r['n_space']}", "Placebo tests", "dates for this meter + comparison meters on the same date", "info")
            + card(str(r["excluded_days"]), "Days excluded", "alert or meter-fault days on this meter", "mute")
            + "</div>")
    T = r["action_date"]
    k = 1000 if RESOURCES[a.resource]["unit"] == "L" else 1
    pre = daily(a.resource, a.building).loc[T - pd.Timedelta(days=56): T - pd.Timedelta(days=1)] / k
    pred, act = r["pred"] / k, r["act"] / k
    sgn = SIGN[a.resource]
    b1 = pred * (1 - sgn * est / 100) / (1 - sgn * lo / 100) if sgn * lo < 100 else pred
    b2 = pred * (1 - sgn * est / 100) / (1 - sgn * hi / 100) if sgn * hi < 100 else pred
    up, dn = np.maximum(b1, b2), np.minimum(b1, b2)
    f1 = go.Figure()
    f1.add_trace(go.Scatter(x=up.index, y=up, line=dict(width=0), showlegend=False, hoverinfo="skip"))
    f1.add_trace(go.Scatter(x=dn.index, y=dn, fill="tonexty", fillcolor="rgba(135,148,171,0.22)", line=dict(width=0),
                            name="Counterfactual range (95%)", hoverinfo="skip"))
    f1.add_trace(go.Scatter(x=pred.index, y=pred, name="Without the action (counterfactual)", line=dict(color="#CBD5E1", dash="dash")))
    f1.add_trace(go.Scatter(x=pre.index, y=pre, name="Actual (before)", line=dict(color="#8794AB")))
    f1.add_trace(go.Scatter(x=act.index, y=act, name="Actual (after)", line=dict(color=RESOURCES[a.resource]["color"], width=2.6)))
    f1.add_vline(x=T, line=dict(color="#34D399", dash="dot"))
    _fig(f1, f"{a.title} · {a.building} — daily {RESOURCES[a.resource]['label']}", 400)
    f2 = go.Figure(go.Histogram(x=np.array(r["null"]), nbinsx=20, marker_color="#3B82F6", opacity=0.75, name="Placebo deviations"))
    f2.add_vrect(x0=lo, x1=hi, fillcolor="rgba(16,185,129,0.12)", line_width=0)
    f2.add_vline(x=est, line=dict(color="#34D399", width=3), annotation_text=f"this action {est:.1f}%",
                 annotation_font_color="#34D399")
    f2.add_vline(x=0, line=dict(color="#8794AB", dash="dot"))
    _fig(f2, "Placebo tests: apparent effects when nothing happened", 400)
    f2.update_xaxes(title="favourable change, % (after bias correction)"); f2.update_yaxes(title="count")
    return card_, f1, f2, comp


# ---------------- EVIDENCE ----------------
def ui_evidence():
    S = AUDIT_SUMMARY
    cards_ = '<div class="gl-cards">' + "".join([
        card(str(sum(len(BLDS[r]) for r in RES_LIST)), "Real meters monitored", f"{len(RES_LIST)} resources · BDG2 site {SITE}", "info"),
        card(str(len(ALERTS)), "Alerts in 6 months", f"chance level if nothing changed ≈ {EXPECTED_FALSE:.0f} window exceedances", "warn"),
        card(str(len(METER_FAULTS)), "Meter fault periods", f"{int(METER_FAULTS.days.sum()) if len(METER_FAULTS) else 0} meter-days excluded", "mute"),
        card(f"{S['naive_shows_benefit']}/{S['tests']}", "Naive 'benefit' at random dates", "before/after > 0 with no action logged", "bad"),
        card(f"{S['verified_benefit'] + S['verified_adverse']}/{S['tests']}", "Certified at random dates",
             "real unexplained changes or chance — no action logged", ""),
    ]) + "</div>"
    return cards_


# ---------------- REPORT & DATA ----------------
def ui_report(as_of_v, window):
    md, paths = build_report(_asof(as_of_v), int(window))
    return md, [paths["html"], paths["md"], paths["csv"], paths["json"]]


def ui_dataset():
    return export_labelled_dataset()


# ---------------- LAYOUT ----------------
HEADER = f"""
<div class="gl-head"><div class="gl-brand"><div class="gl-logo">G</div><div>
<div class="gl-title">GreenLedger</div>
<div class="gl-sub">CAMPUS RESOURCE OPTIMISATION · {" · ".join(RESOURCES[r]["label"].split(" (")[0].upper() for r in RES_LIST)} · IC26-05-01</div></div></div>
<div class="gl-pill">REAL BDG2 METERS · CAMPUS {SITE.upper()} · {sum(len(BLDS[r]) for r in RES_LIST)} METERS · NOTHING SYNTHETIC</div></div>"""

_init_asof = ASOF_CHOICES[-1][1]
_init_window = 42
_ov = ui_overview(_init_asof, _init_window)
_t0, _c0 = _alerts_view(_init_asof, "All", "All")
_a0 = _c0[0][1] if _c0 else None
_f0, _x0 = ui_alert_detail(_a0)
_mf0, _dr0 = ui_faults(_init_asof)

with gr.Blocks(title="GreenLedger") as demo:
    gr.HTML(CSS + HEADER)
    with gr.Row():
        asof_dd = gr.Dropdown(ASOF_CHOICES, value=_init_asof, label="Viewing campus as of", scale=3,
                              info="Alerts and verdicts use only data available that week")
        window_rb = gr.Radio(POST_WINDOWS, value=_init_window, label="Measurement window for verification (days)", scale=2)

    with gr.Tabs() as tabs:
        with gr.Tab("Overview", id="overview"):
            gr.HTML(intro("<b>Every metered resource on one campus.</b> Each meter is compared with what its peers, the "
                          "calendar and the weather say it should read. Colour appears only when a meter does something it "
                          "never did in 18 months of normal operation."))
            ov_cards = gr.HTML(_ov[0])
            gr.HTML('<div class="gl-sec">Resource coverage</div>' + _tbl_html(COVERAGE))
            ov_plot = gr.Plot(_ov[1])
            gr.HTML('<div class="gl-sec">Meter scorecard · selected week</div>')
            ov_score = gr.HTML(_ov[2])

        with gr.Tab("Alerts", id="alerts"):
            gr.HTML(intro("<b>Adverse changes beyond each meter's own record.</b> Tests: 1-hour, 24-hour, 7-day and daily-minimum "
                          "(base load). Staff confirm or dismiss each alert; decisions are logged."))
            with gr.Row():
                res_f = gr.Dropdown(["All"] + RES_LIST, value="All", label="Resource", scale=1)
                st_f = gr.Dropdown(["All", "OPEN", "CONFIRMED", "DISMISSED"], value="All", label="Status", scale=1)
            al_tbl = gr.Dataframe(_t0, label="Alerts up to the selected week", interactive=False, wrap=True)
            al_dd = gr.Dropdown(choices=_c0, value=_a0, label="Inspect alert")
            al_plot = gr.Plot(_f0)
            al_expl = gr.HTML(_x0)
            with gr.Row():
                note_tb = gr.Textbox(label="Note (optional)", scale=3)
                conf_btn = gr.Button("Confirm problem", variant="primary", scale=1)
                dism_btn = gr.Button("Dismiss", scale=1)
            al_msg = gr.HTML()
            gr.HTML('<div class="gl-sec">Decision log</div>')
            dec_tbl = gr.Dataframe(_decision_log(), interactive=False)
            gr.HTML('<div class="gl-sec">Meter data faults (no data or zero readings on an active meter)</div>')
            mf_tbl = gr.Dataframe(_mf0, interactive=False)
            gr.HTML('<div class="gl-sec">Unexplained sustained favourable changes (beyond record, no action logged)</div>')
            dr_tbl = gr.Dataframe(_dr0, interactive=False)

        with gr.Tab("Action Register", id="register"):
            gr.HTML(intro("<b>Log real interventions.</b> The register starts empty: nothing counts until someone logs what was "
                          "done, where and when, and the data verifies it."))
            reg_tbl = gr.Dataframe(_ledger_view(_init_asof, _init_window), label="Action ledger", interactive=False, wrap=True)
            with gr.Row():
                f_title = gr.Textbox(label="What was done", scale=3)
                f_res = gr.Dropdown(RES_LIST, value=RES_LIST[0], label="Resource", scale=1)
                f_bld = gr.Dropdown(BLDS[RES_LIST[0]], value=BLDS[RES_LIST[0]][0], label="Building (metered)", scale=1)
            with gr.Row():
                f_date = gr.Textbox(label="Date (YYYY-MM-DD)", scale=1)
                f_cost = gr.Number(label="Cost ₹ (optional)", value=None, scale=1)
                f_claim = gr.Number(label="Claimed saving % (optional)", value=None, scale=1)
                f_notes = gr.Textbox(label="Notes", scale=2)
            log_btn = gr.Button("Log action", variant="primary")
            log_msg = gr.HTML()

        with gr.Tab("Verify", id="verify"):
            gr.HTML(intro("<b>Did the action work?</b> Counterfactual from untreated comparison meters + placebo tests in time and "
                          "space. Verified only if the 95% interval excludes zero and p ≤ 5%; counted at the lower bound."))
            with gr.Row():
                vf_dd = gr.Dropdown(_action_choices(), value=None, label="Action", scale=4)
                vf_btn = gr.Button("Verify", variant="primary", scale=1)
            vf_card = gr.HTML(intro("Log an action in the <b>Action Register</b>, then select it here."))
            vf_comp = gr.HTML()
            with gr.Row():
                vf_plot1 = gr.Plot()
                vf_plot2 = gr.Plot()

        with gr.Tab("Evidence", id="evidence"):
            gr.HTML(intro("<b>How far to trust the numbers — using real data only.</b> Parameters and their reasons, model accuracy "
                          "on six unseen months, and an audit of the verifier at dates where no action was logged."))
            gr.HTML(ui_evidence())
            gr.HTML('<div class="gl-sec">Parameters — every value and why</div>' + _tbl_html(PARAMETER_TABLE))
            gr.HTML('<div class="gl-sec">Model accuracy on the monitoring period (never seen in training)</div>')
            gr.Dataframe(FIT_DF, interactive=False)
            gr.HTML('<div class="gl-sec">Verifier audit — first day of each month, every meter, no action logged</div>')
            gr.Dataframe(AUDIT_DF, interactive=False, wrap=True)

        with gr.Tab("Report & Data", id="report"):
            gr.HTML(intro("<b>Exportable report</b> (printable HTML → PDF, Markdown, CSV ledger, JSON) and the <b>labelled real "
                          "dataset</b>: BDG2 readings with the platform's alerts, meter faults, staff decisions and verdicts."))
            with gr.Row():
                rep_btn = gr.Button("Generate report", variant="primary")
                ds_btn = gr.Button("Export labelled dataset")
            rep_files = gr.File(label="Report files", file_count="multiple")
            ds_file = gr.File(label="Labelled dataset (ZIP)")
            rep_md = gr.Markdown()

    gr.HTML(f'<div class="gl-foot"><span>GreenLedger · real BDG2 data · placebo-tested verification</span>'
            f'<span>{EF_SOURCE}</span></div>')

    def _refresh_all(as_of_v, window, res_v, st_v):
        c, f, s = ui_overview(as_of_v, window)
        t, dd, fig, ex = ui_alerts(as_of_v, res_v, st_v)
        mf, dr = ui_faults(as_of_v)
        return c, f, s, t, dd, fig, ex, mf, dr, _ledger_view(as_of_v, window)

    _outs = [ov_cards, ov_plot, ov_score, al_tbl, al_dd, al_plot, al_expl, mf_tbl, dr_tbl, reg_tbl]
    asof_dd.change(_refresh_all, [asof_dd, window_rb, res_f, st_f], _outs)
    window_rb.change(_refresh_all, [asof_dd, window_rb, res_f, st_f], _outs)
    res_f.change(ui_alerts, [asof_dd, res_f, st_f], [al_tbl, al_dd, al_plot, al_expl])
    st_f.change(ui_alerts, [asof_dd, res_f, st_f], [al_tbl, al_dd, al_plot, al_expl])
    al_dd.change(ui_alert_detail, [al_dd], [al_plot, al_expl])
    _dec = [al_msg, al_tbl, al_dd, al_plot, al_expl, dec_tbl]
    conf_btn.click(lambda a, n, x, r, s: ui_decide(a, "CONFIRMED", n, x, r, s), [al_dd, note_tb, asof_dd, res_f, st_f], _dec)
    dism_btn.click(lambda a, n, x, r, s: ui_decide(a, "DISMISSED", n, x, r, s), [al_dd, note_tb, asof_dd, res_f, st_f], _dec)
    f_res.change(ui_res_buildings, [f_res], [f_bld])
    log_btn.click(ui_log_action, [f_title, f_res, f_bld, f_date, f_cost, f_claim, f_notes, asof_dd, window_rb],
                  [log_msg, reg_tbl, vf_dd])
    vf_btn.click(ui_verify, [vf_dd, asof_dd, window_rb], [vf_card, vf_plot1, vf_plot2, vf_comp])
    vf_dd.change(ui_verify, [vf_dd, asof_dd, window_rb], [vf_card, vf_plot1, vf_plot2, vf_comp])
    rep_btn.click(ui_report, [asof_dd, window_rb], [rep_md, rep_files])
    ds_btn.click(ui_dataset, None, ds_file)

print("App built.")

## Results summary — please send this output back

In [ ]:
# ============================================================
# 11 · SUMMARY (all numbers computed above from real data)
# ============================================================
print("=" * 78)
print(f"GREENLEDGER · BDG2 campus '{SITE}' · {len(RES_LIST)} resources · {sum(len(BLDS[r]) for r in RES_LIST)} real meters")
print("=" * 78)
print(COVERAGE.to_string(index=False))
print(f"\nMonitoring {EVAL_START:%d %b %Y} → {DATA_END:%d %b %Y}")
print(f"  Alerts: {len(ALERTS)}   Meter-fault periods: {len(METER_FAULTS)}   Unexplained favourable changes: {len(DROPS)}")
print(f"  Verifier audit (no action logged): {AUDIT_SUMMARY}")
print("\nModel accuracy (median per resource, % of each meter's typical level):")
print(FIT_DF.groupby("Resource")[["Peers in model", "Bias % of typical", "Daily error % of typical"]].median().round(1).to_string())

## Launch

In [ ]:
# ============================================================
# 12 · LAUNCH  (Colab / Kaggle: prints a public link; Kaggle needs Internet ON)
# ============================================================
if os.environ.get("GL_NO_LAUNCH") != "1":
    try:
        demo.queue().launch(share=True, debug=False)
    except Exception as ex:
        print("Public link unavailable:", ex)
        demo.queue().launch(share=False, debug=False)